In [ ]:
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

backbone_model_id = "Qwen/Qwen3-4B"
peft_id = "erin99/Qwen3-4B-SquadV2-LoRA"  # '/home/lxm/workspace/Shadow/outputs/squad_v2_suite_lora_4B/squad_v2_squad_v2/lora'
tokenizer = AutoTokenizer.from_pretrained(backbone_model_id)
base_model = AutoModelForCausalLM.from_pretrained(backbone_model_id).to('cuda:6')

model = PeftModel.from_pretrained(
    base_model, peft_id, is_trainable=False).to(base_model.device)

model = model.eval()



/home/lxm/miniconda3/envs/py311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 3/3 [00:00<00:00,  3.05it/s]


In [2]:
import torch

# Example GSM8K problem
# question = "Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?"
question = "1+1=?"
question = "Ariel was playing basketball. 1 of her shots went in the hoop. 2 of her shots did not go in the hoop. How many shots were there in total?"
question = 'our team scored a total of 123 points. 67 points were scored in the first half. How many were scored in the second half?'
question = "小王的妈妈有三个儿子，大儿子叫大明，二儿子叫二明，三儿子叫什么？"
# question = "9.3 and 9.200 which is bigger?"
question = "blood relationship identify: Grand Mother's Brother"
question = '有一串彩珠，按“2红3绿4黄”的顺序依次排列。第509颗是什么颜色？'

user_content = f"Question: {question}"
user_message = {"role": "user", "content": user_content}
prompt = tokenizer.apply_chat_template(
    [user_message],
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True,
)

# Tokenize and generate
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=True,
    )

# Decode and print the result
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(generated_text)


user
Question: 有一串彩珠，按“2红3绿4黄”的顺序依次排列。第509颗是什么颜色？
Answer:
assistant
<think>
嗯，我现在要解决这个问题，有一串彩珠，按“2红3绿4黄”的顺序依次排列。第509颗是什么颜色？首先，我需要理解题目的意思，然后找出规律，最后计算第509颗珠子的颜色。

首先，题目说彩珠是按照“2红3绿4黄”的顺序依次排列的。也就是说，每一轮重复的顺序是2红3绿4黄，对吧？那这组颜色的顺序是红红绿绿绿黄黄黄黄？不对，应该是2红3绿4黄，所以每一轮的颜色数目是2+3+4=9颗珠子。对吗？比如第一轮是2红，然后3绿，再4黄，所以每一轮是9颗珠子，然后循环往复。

那第509颗珠子是什么颜色呢？我需要确定第509颗珠子处于哪一个循环周期中的哪一个位置。

首先，我需要计算509除以9，得到的商和余数。因为每一轮是9颗珠子，所以商就是循环的次数，余数就是位置。

不过，我需要确认一下，余数为0的话
